# NCF for movielens

In [1]:
import os, json, bson

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf

2023-07-28 18:14:18.256146: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-28 18:14:18.438554: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-07-28 18:14:19.131162: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2023-07-28 18:14:19.131263: W tensorflow/stream_

# Data Load 

In [2]:
movielens_dir = '../data/ml-100k/'

In [3]:
df = pd.read_csv(os.path.join(movielens_dir, 'u.data'), sep='\t', header=None)
df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
print(df.shape)
df.head()

(100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [4]:
all_users = list(set(df['user_id']))
all_items = list(set(df['item_id']))

len(all_users), len(all_items)

(943, 1682)

# Prepare Train/Test data

- leave-one-out evaluation (positive)
    - 테스트 데이터: 유저의 latest 데이터
    - 학습 데이터: 유저의 나머지 데이터
- 학습 시에는 모든 negative 데이터를 포함
- 평가 시에는 positive test data 하나와 나머지 샘플링한 unobserved 100개 데이터를 합하여 랭킹
- 위 랭킹 결과의 top k에 대해서 positive 가 포함되어 있으면 hit, 그 위치가 어디인지를 NDCG가 측정

In [5]:
df.sort_values(by=['user_id', 'timestamp'], ascending=[True, False], inplace=True)
df.head()

,user_id,item_id,rating,timestamp
3248,1,74,1,889751736
19699,1,102,2,889751736
30479,1,256,4,889751712
47638,1,5,3,889751712
687,1,171,5,889751711


In [6]:
positive_data_per_user = df

In [9]:
all_users = df['user_id'].unique()
all_items = df['item_id'].unique()
len(all_users), len(all_items)

(943, 1682)

### filter users having at least 20 interactions

In [16]:
all_users = all_users[df.groupby(by='user_id').count()['item_id'] > 20]
len(all_users)

911

In [17]:
print('before filtering shape: ', df.shape)
df = df[df['user_id'].apply(lambda x: x in all_users)]
print('after filtering shape: ', df.shape)

before filtering shape:  (100000, 4)
after filtering shape:  (99360, 4)


전체 데이터 수 확인, pos, neg 데이터 분배

In [18]:
movie_map = {p:i for i,p in enumerate(all_items)}
user_map = {u:i for i,u in enumerate(all_users)}

len(movie_map.keys()), len(user_map.keys())

(1682, 911)

In [23]:
data_pos = [[id_] for id_ in all_users] # user, item, score

for i, row in df.iloc[::-1].iterrows():
    data_pos[user_map[row['user_id']]].append(movie_map[row['item_id']])

In [ ]:
num_negative = 100

In [32]:
train_pos = []
test_pos = []
avg_pos = 0

for history in tqdm(data_pos):
    avg_pos += len(history)-1
    test_pos.append(history[-1])
    train_pos.append(history[1:-1])

100%|██████████| 911/911 [00:00<00:00, 456458.12it/s]


In [33]:
print('average history length: ', avg_pos/len(all_users))

average history length:  109.06695938529089


In [27]:
def sampling(num_sample, without):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.randint(0, 5043)
        if all_pins[randint] in without:
            continue
        else:
            # sampled.append([user_map[user_board], pin_map[all_pins[randint]], 0])
            sampled.append(item_id)
            idx += 1
    return sampled

for i, user in enumerate(train_pos):
    train_pos[i].extend(sampling(num_negative, movie_map.keys(), b['pins']))
    test_pos[i].extend(sampling(100, movie_map.keys(), b['pins']))

NameError: name 'num_negative' is not defined

* pinterest -> user: item1, item2, ...
* movielens -> (user, item), (user, item) (user, item)

거꾸로 돌면서 첫번째 등장한 데이터를 테스트 나머지를 트레인으로 두기
양의 데이터는 [user_id, item_id]

In [8]:
# train = [] # user, item, score
# test = [] # user, item, score

# for b in tqdm(board_pins):
#     train.extend([[user_map[b['board_id']], pin_map[p], 1] for p in b['pins'][:-1]])
#     test.append([user_map[b['board_id']], pin_map[b['pins'][-1]], 1])

100%|██████████| 46000/46000 [00:06<00:00, 7162.25it/s] 


In [9]:
def sampling(num_sample, all_pins, without, user_board):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.randint(0, 5043)
        if all_pins[randint] in without:
            continue
        else:
            sampled.append([user_map[user_board], pin_map[all_pins[randint]], 0])
            idx += 1
    return sampled

for b in tqdm(board_pins):
    train.extend(sampling(num_negative, all_pins, b['pins'], b['board_id']))
    test.extend(sampling(100, all_pins, b['pins'], b['board_id']))

100%|██████████| 46000/46000 [00:36<00:00, 1265.02it/s]


In [10]:
train, test = np.array(train), np.array(test)
train.shape, test.shape

((5049242, 3), (4646000, 3))

In [11]:
np.random.shuffle(train)
np.random.shuffle(test)

샘플링은 유저별로 55번씩하면 되는데 positive에 있는 데이터가 들어가면 안됨

In [12]:
all_users = {tp[0] for tp in np.concatenate([train,test], axis=0)}
all_items = {tp[1] for tp in np.concatenate([train,test], axis=0)}
len(all_users), len(all_items)

(46000, 2565241)

# GMF (generalization matrix factorization)

## build

In [13]:
def get_gmf(num_users, num_items, latent_dim):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]))([user_flatten, item_flatten])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [14]:
# hyperparams
learning_rate = 0.001


mirrored_strategy = tf.distribute.MirroredStrategy()

with mirrored_strategy.scope():
  gmf_model = get_gmf(len(all_users), len(all_items), 16)

gmf_model.summary()

2023-07-20 07:55:51.301029: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.302811: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.314126: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.315723: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.317322: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 user_embedding (Embedding)     (None, 1, 16)        736000      ['input_1[0][0]']                
                                                                                                  
 item_embedding (Embedding)     (None, 1, 16

In [15]:
gmf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy")

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


## training

In [16]:
gmf_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

Epoch 1/10
INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to

## prediction

In [17]:
gmf_test_pred = gmf_model.predict([test[:, 0], test[:, 1]], batch_size=2048)

2269/2269 [==============================] - 8s 3ms/step


## Multi Layer Perceptron

In [ ]:
def get_mlp(num_users, num_items, latent_dim, layer_dims):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Concatenate()([user_flatten, item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    # last layer
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [ ]:
mlp_model = get_mlp(len(all_users), len(all_items), 16, [32, 16, 8])
mlp_model.summary()

In [ ]:
learning_rate = 0.00001
mlp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

In [ ]:
mlp_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

# NeuMF (Neural Matrix Factorization)

In [ ]:
def get_neuMF(num_users, num_items, latent_dim, layer_dims):

    # common inputs
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    # gmf branch
    gmf_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='gmf_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    gmf_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='gmf_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    gmf_user_flatten = tf.keras.layers.Flatten()(gmf_user_embedding)
    gmf_item_flatten = tf.keras.layers.Flatten()(gmf_item_embedding)

    gmf_multiply = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]), name='gmf_multiply')(
        [gmf_user_flatten, gmf_item_flatten])

    # mlp branch
    mlp_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='mlp_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    mlp_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='mlp_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    mlp_user_flatten = tf.keras.layers.Flatten()(mlp_user_embedding)
    mlp_item_flatten = tf.keras.layers.Flatten()(mlp_item_embedding)

    x = tf.keras.layers.Concatenate()([mlp_user_flatten, mlp_item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
        
    # concat branches
    x = tf.keras.layers.Concatenate()([gmf_multiply, x])

    # last layer
    neumf_last_layer = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42), bias_constraint=None)(x)

    neumf_model = tf.keras.Model([user_input, item_input], neumf_last_layer)


In [ ]:
neuMF_model = get_neuMF(len(all_users), len(all_items), 16, [32, 16, 8])
neuMF_model.summary()

## load saved weights

In [ ]:
learning_rate = 0.00001
neuMF_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

## Training

In [ ]:
neuMF_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

# Evaluation

## NDCG

In [18]:
gmf_test_pred

array([[3.2802136e-06],
       [4.4711422e-05],
       [3.6975984e-05],
       ...,
       [5.1606380e-06],
       [1.1082555e-07],
       [1.2884149e-06]], dtype=float32)

In [33]:
import imp
import evaluate
imp.reload(evaluate.ndcg)

<module 'evaluate.hr' from '/home/leejuyeon/etc/recsys/models/../evaluate/hr.py'>

In [23]:
import sys
sys.path.insert(0, '..')
from evaluate import ndcg #import average_ndcg

topn = 10
ndcg.average_ndcg(test[:, 0], gmf_test_pred, test[:, 2], topn, len(all_users))

0.9981521739130435

## Hit rate

In [ ]:
imp.reload(evaluate.hr)

In [34]:
from evaluate import hr

hr, hits = hr.average_hr(test[:, 0], gmf_test_pred, test[:, 2], topn, len(all_users))

In [40]:
np.unique(hits, return_counts=True)

(array([0, 1]), array([   85, 45915]))